# Nepali Summarization — mT5-small + LoRA (Baseline)

Run on **Google Colab** with a GPU runtime: `Runtime > Change runtime type > T4 GPU`.
**Do NOT run this locally** — it needs a CUDA GPU and current PyTorch.

This is the Phase-2 baseline: fine-tune `google/mt5-small` with LoRA on Nepali article→summary pairs, then score with ROUGE.

In [ ]:
# datasets 4.0+ removed script-based loaders that XL-Sum needs, so pin 2.21.0.
!pip -q install transformers "datasets==2.21.0" "fsspec<=2024.6.1" peft evaluate rouge_score sentencepiece accelerate

In [ ]:
# Load the Nepali subset of XL-Sum directly (no upload needed).
from datasets import load_dataset
ds = load_dataset('csebuetnlp/xlsum', 'nepali', trust_remote_code=True)
print(ds)
print(ds['train'][0]['summary'][:200])

In [ ]:
from transformers import AutoTokenizer
MODEL = 'google/mt5-small'
tok = AutoTokenizer.from_pretrained(MODEL)

MAX_IN, MAX_OUT = 512, 64  # summary task; raise MAX_OUT for longer summaries

def preprocess(batch):
    x = tok(batch['text'], max_length=MAX_IN, truncation=True)
    y = tok(text_target=batch['summary'], max_length=MAX_OUT, truncation=True)
    x['labels'] = y['input_ids']
    return x

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)

In [ ]:
from transformers import AutoModelForSeq2SeqLM
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM, r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=['q', 'v'],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
import numpy as np, evaluate
from transformers import (DataCollatorForSeq2Seq, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments)

rouge = evaluate.load('rouge')
collator = DataCollatorForSeq2Seq(tok, model=model)

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    dp = tok.batch_decode(preds, skip_special_tokens=True)
    dl = tok.batch_decode(labels, skip_special_tokens=True)
    return rouge.compute(predictions=dp, references=dl, use_stemmer=False)

args = Seq2SeqTrainingArguments(
    output_dir='mt5-ne-lora',
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=MAX_OUT,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized['train'], eval_dataset=tokenized['validation'],
    data_collator=collator, compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
# Final test-set ROUGE
print(trainer.evaluate(tokenized['test']))

In [ ]:
# Eyeball a few generated summaries
import torch
model.eval()
for ex in ds['test'].select(range(3)):
    ids = tok(ex['text'], return_tensors='pt', truncation=True, max_length=MAX_IN).input_ids.to(model.device)
    out = model.generate(ids, max_length=MAX_OUT, num_beams=4)
    print('PRED:', tok.decode(out[0], skip_special_tokens=True))
    print('GOLD:', ex['summary'][:200])
    print('-' * 60)